# Service Kernel Gateway — CFR OpenSpiel

Service du **pli 0** de l'Epic #19571 : exposer le carnet phare
[GameTheory-13-ImperfectInfo-CFR-Python.ipynb](../GameTheory-13-ImperfectInfo-CFR-Python.ipynb)
en microservice REST via Jupyter **Kernel Gateway** (mode `notebook-http`).

**Contrat** : `POST /gametheory/cfr` — corps `{"game": "kuhn_poker", "iterations": 300}`
-> `{"status": "ok", "game": ..., "iterations": ..., "exploitability": ..., "nash_conv": ..., "policy": {...}}`.

Jeux supportes : `kuhn_poker`, `leduc_poker`, `matrix_rps` (chargé via la transformation
`turn_based_simultaneous_game`, CFR exigeant un arbre séquentiel).

Ce carnet s'exécute aussi **hors gateway** (mode local) : il sert alors de démonstration
sur `kuhn_poker`. Voir le [README](README.md) pour le démarrage.


In [1]:
# Cellule de preparation — executee une fois au demarrage du kernel.
import json
import os
import sys

# Le kernel demarre dans ce dossier (seed_uri) : il y trouve openspiel_cfr_core.py.
sys.path.insert(0, os.getcwd())

from openspiel_cfr_core import solve_cfr

# REQUEST n'existe que sous Kernel Gateway (mode notebook-http) ; l'objet est
# injecte au moment de chaque requete. Hors gateway : None -> mode local.
REQUEST = globals().get("REQUEST")

# Jeu de demonstration (carnet phare GameTheory-13 : poker de Kuhn).
DEMO_GAME = "kuhn_poker"
DEMO_ITERATIONS = 300

print("Mode :", "Kernel Gateway (notebook-http)" if REQUEST is not None else "local (demonstration)")


Mode : local (demonstration)


## Endpoint `POST /gametheory/cfr`

La cellule ci-dessous est annotée en première ligne par `# POST /gametheory/cfr` :
Kernel Gateway l'exécute à chaque requête HTTP, avec l'objet `REQUEST` injecté
dans l'espace de noms. Le JSON imprimé devient le corps de la réponse.


In [2]:
# POST /gametheory/cfr
# Sous Kernel Gateway, REQUEST porte la requete : chaine JSON en v3
# (double json.dumps cote serveur), dict direct selon les versions -- on
# accepte les deux. Le corps JSON (Content-Type application/json) est
# deja parse dans la cle "body".
requete = None
if isinstance(REQUEST, str):
    try:
        requete = json.loads(REQUEST)
    except Exception:
        requete = None
elif isinstance(REQUEST, dict):
    requete = REQUEST
corps = requete.get("body") if isinstance(requete, dict) else None
game = corps.get("game") if isinstance(corps, dict) else None
iterations = corps.get("iterations") if isinstance(corps, dict) else None
if REQUEST is None:
    # Execution locale : demonstration sur le jeu du carnet phare.
    game = DEMO_GAME
    iterations = DEMO_ITERATIONS
reponse = solve_cfr(game, iterations)
print(json.dumps(reponse))


{"status": "ok", "game": "kuhn_poker", "iterations": 300, "exploitability": 0.003034, "nash_conv": 0.006068, "policy": {"0": {"Pass": 0.811179, "Bet": 0.188821}, "1p": {"Pass": 0.988333, "Bet": 0.011667}, "0pb": {"Pass": 0.998973, "Bet": 0.001027}, "1b": {"Pass": 0.658747, "Bet": 0.341253}, "2p": {"Pass": 0.003333, "Bet": 0.996667}, "2b": {"Pass": 0.001667, "Bet": 0.998333}, "1": {"Pass": 0.975026, "Bet": 0.024974}, "0p": {"Pass": 0.665731, "Bet": 0.334269}, "1pb": {"Pass": 0.470903, "Bet": 0.529097}, "0b": {"Pass": 0.998333, "Bet": 0.001667}, "2": {"Pass": 0.431393, "Bet": 0.568607}, "2pb": {"Pass": 0.001932, "Bet": 0.998068}}}
